# Workbook 4 — "Inductive bias" is a four-letter word

**Exercises · Mysteries of Deep Learning, Iliad Intensive 2026**

"Neural networks generalize because of their inductive bias" is true, and as an
explanation it is empty: every learner that generalizes has an inductive bias. This
workbook works through the textbook chain from Occam's razor to Kolmogorov complexity
to Solomonoff's universal prior, then points to current accounts of what deep networks
prefer: short programs, small circuits, small norms, and structure in the data itself.

Route (sections are largely independent; the math is in §2–3 and in the exercises of
§4–5):

- [1. Inductive bias is a curiosity stopper](#wb4-section-1). Two sentences of orientation.
- [2. Occam, two ways](#wb4-section-2). Bayesian evidence (prior volume), Kolmogorov complexity (description length) and MDL two-part codes, and where they part. Derivation exercises: the Occam factor of a Gaussian model, the four basic facts about $K$, why MDL states parameters to precision $1/\sqrt n$, and two on where the notions disagree.
- [3. Solomonoff](#wb4-section-3). The universal prior and its $K(\mu)\ln 2$ regret bound, and Legg's limits on computable predictors, both derived as exercises; a pointer to simplicity bias in random networks. Core reading: Rathmanner and Hutter.
- [4. Program synthesis and the circuit prior](#wb4-section-4). Two readings (Furman's program-synthesis hypothesis; Hänni and Vaintrob's circuit-prior theorem) and one exercise: sparse parity for a counting learner versus a kernel learner.
- [5. Hierarchical representations](#wb4-section-5). The Random Hierarchy Model, with one estimation exercise.
- [6. The accounts side by side](#wb4-section-6). One table of the priors over functions, and where singular learning theory sits.

The notebook ends with links to related workbooks and further reading.

**How to use this notebook.** Pick what interests you; nobody is expected to do
everything. The exercises are mostly derivations with hints. Worked solutions are in
the solutions version of this notebook, which everyone may use. Each reading states its
result first so you can decide whether to open it; *core* marks the one to read if
you read only one in that section.

<a id="wb4-section-1"></a>

## 1. Inductive bias is a curiosity stopper

By no-free-lunch (WB2 §2), a learner that does well on the tasks we meet must favour
some targets over others; "inductive bias" names that preference without saying what
it is. §2–5 give candidate answers: prior volume and description length (§2–3),
program length, circuit size and kernel norm (§4), and structure in the data (§5).

<a id="wb4-section-2"></a>

## 2. Occam, two ways

**The result.** Bayesian model comparison penalizes a model by how much prior volume
it wastes on data that did not occur: the Occam factor. Kolmogorov complexity
penalizes a hypothesis by the length of its shortest description on a fixed universal
machine. MDL's two-part code sits between them. Evidence and two-part code agree at
leading order for regular parametric models and come apart in instructive cases, so
"simplicity" is not one thing.

### 2.A. Bayesian Occam: volume

[Source: MacKay, *Bayesian Interpolation* (1992).](https://authors.library.caltech.edu/records/r7qgh-q6g10)

A model $\mathcal M$ has parameters $\theta\in\mathbb R^p$, a prior $\pi(\theta)$
and a likelihood $p(D\mid\theta)$. Its *evidence* (marginal likelihood) is

$$
Z(\mathcal M)=\int p(D\mid\theta)\,\pi(\theta)\,d\theta,
$$

and two models are compared by $Z_1/Z_2$ times the prior odds.

With one parameter, a prior of width $\Delta_\pi$ and a likelihood sharply peaked at
$\hat\theta$ with width $\Delta_{\text{post}}$,

$$
Z\approx p(D\mid\hat\theta)\cdot\frac{\Delta_{\text{post}}}{\Delta_\pi}.
$$

The second factor, at most 1, is the **Occam factor**: the fraction of the prior's
bets that turned out compatible with the data. A flexible model spreads its prior
over many datasets it could have fit, so it put little probability on whichever one
arrived. Nothing here refers to how long the model is to write down.

With $p$ parameters and $n$ data points, the Laplace approximation gives
$-\ln Z=-\ln p(D\mid\hat\theta)+\tfrac p2\ln n+O(1)$, which is BIC (WB2 §7). For
singular models (the SLT day), $\tfrac p2$ is replaced at leading order by the
learning coefficient $\lambda\le\tfrac p2$. Either way the penalty measures the
*volume* of near-optimal parameters, not the parameter count.

#### 2.A.i. The Occam factor, exactly

Data $y_1,\dots,y_n$ are i.i.d. $\mathcal N(\theta,\sigma^2)$ with $\sigma$ known,
and the prior is $\theta\sim\mathcal N(0,\tau^2)$. Write $\bar y$ for the sample mean
and $s^2=\sigma^2/n$.

1. Show that $\sum_i(y_i-\theta)^2=\sum_i(y_i-\bar y)^2+n(\theta-\bar y)^2$, and hence
   $p(D\mid\theta)=p(D\mid\bar y)\,e^{-(\theta-\bar y)^2/2s^2}$, so $\hat\theta=\bar y$.
2. Show that
   $Z=p(D\mid\hat\theta)\cdot\sqrt{\dfrac{s^2}{\tau^2+s^2}}\,\exp\!\Big(-\dfrac{\bar y^2}{2(\tau^2+s^2)}\Big)$.
3. Show that for large $n$,
   $-\ln Z=-\ln p(D\mid\hat\theta)+\tfrac12\ln n+\tfrac12\ln\dfrac{\tau^2}{\sigma^2}+\dfrac{\bar y^2}{2\tau^2}+o(1)$,
   and match the terms with $\Delta_{\text{post}}/\Delta_\pi$.
4. Fix the data and let $\tau\to\infty$, an ever vaguer prior. What happens to $Z$,
   and why does this not mean the model fits worse?

<details><summary>Hint</summary>

For step 2, write $e^{-(\theta-\bar y)^2/2s^2}=\sqrt{2\pi s^2}\;\mathcal N(\bar y;\theta,s^2)$.
Integrating the density $\mathcal N(\theta;0,\tau^2)$ against
$\mathcal N(\bar y;\theta,s^2)$ over $\theta$ gives $\mathcal N(\bar y;0,\tau^2+s^2)$,
the density of a sum of two independent Gaussians. For the matching in step 3, take
$\Delta_\pi=1/\pi(\hat\theta)$, the width of a flat prior with the same density at
$\hat\theta$.
</details>

### 2.B. Description length: Kolmogorov complexity

Fix a universal prefix machine $U$. Its valid programs form a prefix-free set of
binary strings: no program is a proper prefix of another. The (prefix) Kolmogorov
complexity of a finite string $x$ is

$$
K_U(x)=\min\{\,|p| : U(p)=x\,\}.
$$

Four basic facts, proved in Exercise 2.B.i:

- **Kraft.** If $P$ is prefix-free then $\sum_{p\in P}2^{-|p|}\le1$. Hence
  $\sum_x2^{-K(x)}\le1$, so $2^{-K(x)}$ is a sub-probability over strings: "short
  description" and "high prior probability" are the same thing in two units.
- **Invariance.** For two universal machines $U,V$ there is a constant $c$ with
  $|K_U(x)-K_V(x)|\le c$ for all $x$. Asymptotic statements do not depend on the
  machine, but for a single finite object the constant can exceed the object's
  complexity, so "the Kolmogorov complexity of this trained network" has no
  machine-independent value.
- **Counting.** Fewer than a $2^{-c}$ fraction of $n$-bit strings have $K(x)<n-c$.
  Simplicity is rare, which is why a preference for it carries information.
- **Uncomputability.** $K$ is not computable, though it can be approximated from above.

[Rathmanner and Hutter §5.9](https://arxiv.org/html/1105.5721) (the core reading of §3) gives an informal introduction,
including the reference-machine problem. The proofs below are the standard textbook
arguments.

#### 2.B.i. The four facts

**(a) Kraft.** To a binary string $p=p_1\dots p_k$ assign the half-open interval
$I_p=[0.p_1\dots p_k,\ 0.p_1\dots p_k+2^{-k})\subset[0,1)$, reading $0.p_1\dots p_k$
as a binary fraction. Show that $I_p\cap I_q=\varnothing$ when neither of $p,q$ is a
prefix of the other. Conclude Kraft's inequality, and then $\sum_x2^{-K_U(x)}\le1$.

**(b) Invariance.** Universality of $U$ means that for every prefix machine $V$ there
is a string $s_V$ with $U(s_Vp)=V(p)$ for all $p$. Prove $K_U(x)\le K_V(x)+|s_V|$.
Then, for a fixed string $x^\star$, construct a universal prefix machine $V$ with
$K_V(x^\star)=1$. What does this say about the size of the invariance constant?

**(c) Counting.** Bound the number of programs shorter than $n-c$ bits, and conclude
the counting statement.

**(d) Uncomputability (the Berry argument).** Suppose $K$ were computable. Build a
program $q_m$ that outputs a string of complexity at least $m$, and bound $|q_m|$.
Derive a contradiction. Then explain how $K(x)$ can nevertheless be approximated from
above, and why you can never be sure the approximation has converged.

<details><summary>Hints</summary>

(a) If neither string is a prefix of the other, they first differ at some position $j$.
Look at which half of a dyadic interval of length $2^{-(j-1)}$ each one lies in.

(b) Let $V$ reserve one very short program for $x^\star$ and otherwise defer to $U$.
Check that $V$'s programs are still prefix-free.

(c) How many binary strings have length less than $n-c$? Each program outputs at most
one string.

(d) To write $m$ in a self-delimiting way, write each bit of its binary expansion twice
and end with `01`. That costs about $2\log_2m$ bits.
</details>

### 2.C. MDL: the two-part code

[Source: Grünwald, *A Tutorial Introduction to the Minimum Description Length Principle*, §2.4.2.](https://arxiv.org/html/math/0406077)

Minimum description length fixes a class of codes in advance and scores a hypothesis
$H$ by $L(H)+L(D\mid H)$: the bits to describe $H$, plus the bits to describe the
data using $H$. With $L(D\mid H)=-\log_2p(D\mid H)$ and $L(H)$ from a prefix code,
this is $-\log_2$ of a prior times a likelihood, so it is a MAP criterion with prior
$2^{-L(H)}$. For a continuous parameter you must also decide how precisely to state
it. Exercise 2.C.i shows that the best precision is of order $1/\sqrt n$ and costs
about $\tfrac12\log_2n$ bits per parameter, which recovers BIC at leading order. MDL
is computable by construction, because you chose the code class. Rathmanner and
Hutter §9.2 places MDL relative to Solomonoff induction.

#### 2.C.i. Why precision $1/\sqrt n$?

One parameter $\theta$ ranges over an interval of length $\Delta_\pi$. Near the
maximum-likelihood estimate,
$-\ln p(D\mid\theta)\approx-\ln p(D\mid\hat\theta)+\tfrac{nJ}{2}(\theta-\hat\theta)^2$,
where $J$ is the Fisher information per sample. Assume sender and receiver both know
$n$ and $J$. The code rounds $\hat\theta$ to the nearest point $\theta_q$ of a grid
with spacing $\delta$, sends its index ($\log_2(\Delta_\pi/\delta)$ bits), and then
sends the data using the code $p(\cdot\mid\theta_q)$.

1. Averaging over where $\hat\theta$ falls in its grid cell, $\theta_q-\hat\theta$ is
   uniform on $[-\delta/2,\delta/2]$. Write the expected total length $L(\delta)$ in bits.
2. Minimize $L(\delta)$ over $\delta$.
3. Show $L(\delta^\ast)=-\log_2p(D\mid\hat\theta)+\tfrac12\log_2n+O(1)$, and say what
   the $O(1)$ depends on.
4. Why does the optimal $\delta$ shrink like $1/\sqrt n$?

<details><summary>Hint</summary>

$\mathbb E[(\theta_q-\hat\theta)^2]=\delta^2/12$. Convert nats to bits by dividing by $\ln2$.
</details>

### 2.D. Where they part

| | penalizes | depends on | computable? |
|---|---|---|---|
| Bayesian Occam | prior volume wasted on data that did not occur | the prior and likelihood | yes, as an integral |
| Kolmogorov | length of the shortest description | the universal machine, up to an additive constant | no |
| MDL two-part | description of hypothesis plus data given hypothesis | the chosen code class and the parameter coordinates | yes |

In a regular parametric model with a smooth prior, evidence and two-part code both
give about $\tfrac p2\log n$ (Exercise 2.A.i and Exercise 2.C.i). They are still
different mechanisms that happen to agree there. *Ideal* MDL, the two-part code with
$L(H)=K(H)$, inherits Kolmogorov's uncomputability. When a paper says "networks are
biased toward simple functions", it is worth asking which of the three it means.

#### 2.D.i. Make them disagree

Give three cases where the volume and description-length notions of simplicity
disagree:

1. a model that is short to describe but heavily Occam-penalized;
2. a hypothesis that is short to describe but that Bayesian model comparison over a
   reasonable model family will not favour;
3. the reverse direction: a model that a parameter-counting code calls expensive but
   that the evidence barely penalizes.

<details><summary>Hint</summary>

For 1, keep one parameter but change the prior (Exercise 2.A.i, step 4). For 2, think
of the digits of $\pi$. For 3, think of a singular model from the SLT day.
</details>

#### 2.D.ii. Reparametrize

Let $\varphi=g(\theta)$ be a smooth bijection. Which of these change: the evidence
$Z$; the MAP estimate and the function $f_{\hat\theta}$ it picks; the two-part code
length; $K$ of the fitted function? What does your answer say about "flat minima
generalize" arguments stated in parameter coordinates?

<details><summary>Hint</summary>

The prior density in $\varphi$-coordinates is $\pi_\varphi(\varphi)=\pi(\theta(\varphi))\,|d\theta/d\varphi|$.
The likelihood is a function, not a density, so it picks up no Jacobian. For flatness,
compute $d^2L/d\varphi^2$ at a critical point of $L$.
</details>

<a id="wb4-section-3"></a>

## 3. Solomonoff

**The result.** Solomonoff's universal prior weights each program by $2^{-|p|}$ and
predicts a binary sequence by mixing everything the programs output. Against any
computable environment (measure) $\mu$, its cumulative log-loss exceeds $\mu$'s own by at most
$K(\mu)\ln 2$ nats (plus a constant), on every sequence and for every length. §3.C
lists what it does not say.

**[A philosophical treatise of universal induction](https://arxiv.org/abs/1105.5721)**, Samuel Rathmanner and Marcus Hutter, 2011 (*Entropy* 13(6), 1076–1136) · *core*

An accessible exposition of Solomonoff induction for non-specialists: Bayes mixtures and their bounds, Kolmogorov complexity, the universal prior and its prediction bounds, computable approximations, and the philosophical objections. About 60 pages. For this notebook: §4.7–4.9 (Bayes mixtures), §5.9 (Kolmogorov complexity), §6.5–7.1 (the universal prior), §8.1 (total bounds), §9 (MDL and computable approximations) and §10.2 (dependence on the reference machine).

### 3.A. The universal prior

Use a *monotone* universal machine $U$. It reads program bits one at a time and
writes output bits it never erases. For a finite binary string $x$, define

$$
M(x)=\sum_{p\ \text{minimal}:\ U(p)\ \text{begins with}\ x} 2^{-|p|},
$$

where "minimal" means no proper prefix of $p$ already outputs something beginning
with $x$. Operationally: feed $U$ fair coin flips as its program; $M(x)$ is the
probability that the output begins with $x$. Short programs get most of the weight.

$M$ is a **semimeasure**, not a measure: $M(\epsilon)\le1$ and
$M(x)\ge M(x0)+M(x1)$, with strict inequality in general, because some programs
print $x$ and then halt, or run forever without printing another bit. $M$ is *lower
semicomputable*: run all programs in parallel and the running sums increase toward
$M(x)$. It is not computable, since you cannot know how much mass is still to come.

### 3.B. Dominance and the regret bound

For a lower-semicomputable $\nu$, $K(\nu)$ denotes the length of a shortest prefix
program that lower-approximates its values, rather than necessarily computing them
exactly. For a computable measure $\mu$ this agrees, up to an additive constant,
with the length of the shortest program that computes $\mu$; the Theorem below uses
the first definition. It is convenient to work with the mixture

$$
\xi(x)=\sum_{\nu} 2^{-K(\nu)}\,\nu(x),
$$

summed over all lower-semicomputable semimeasures $\nu$; the weights sum to at most 1
by Kraft. $\xi$ and $M$ agree up to multiplicative constants (Rathmanner and Hutter
§7–8), so everything below holds for $M$ with $K(\mu)$ replaced by $K(\mu)+O(1)$.
Predict with the conditionals $\xi(a\mid x)=\xi(xa)/\xi(x)$.

**Theorem.** For every computable measure $\mu$, every $n$, and every $x_{1:n}$ with
$\mu(x_{1:n})>0$,

$$
\sum_{t=1}^{n}\Big[-\ln\xi(x_t\mid x_{<t})\Big]-\sum_{t=1}^{n}\Big[-\ln\mu(x_t\mid x_{<t})\Big]
\ \le\ K(\mu)\ln 2 .
$$

This is a bound on the mixture forecaster's cumulative online log-loss, not on any
single hypothesis. Taking expectations under $\mu$, the expected relative entropies
between $\mu$'s and $\xi$'s next-symbol predictions (defined in Exercise 3.B.i(c),
where $\xi$'s conditionals may sum to less than 1) sum, over all time, to at most
$K(\mu)\ln2$, so they tend to zero and the mixture's predictions converge to the truth.
Rathmanner and Hutter §8.1 states the squared-error version and the sharpening to
convergence with $\mu$-probability one. Exercise 3.B.i derives the bound.

This is the Bayes-mixture bound: any Bayesian mixture that gives the truth prior
weight $w_\mu$ has log-loss regret at most $\ln(1/w_\mu)$ (Rathmanner and Hutter
§4.7–4.9). Solomonoff's contribution is the class, which contains every computable
environment, and the weights $2^{-K}$, which make the price "the length of the truth's
shortest description."

#### 3.B.i. Derive the regret bound

Take a countable class of semimeasures $\nu$ with weights $w_\nu>0$,
$\sum_\nu w_\nu\le1$, and let $\xi=\sum_\nu w_\nu\nu$. Suppose the class contains a
measure $\mu$ that generates the data.

**(a)** Show that $\xi(x)\ge w_\mu\,\mu(x)$ for every $x$.

**(b)** Show that $\prod_{t\le n}\xi(x_t\mid x_{<t})=\xi(x_{1:n})/\xi(\epsilon)$, and
conclude that the cumulative log-loss regret of $\xi$ against $\mu$ is at most
$\ln(1/w_\mu)$ on every sequence. Specialize to $w_\nu=2^{-K(\nu)}$.

**(c)** Let $d_t(x_{<t})=\sum_a\mu(a\mid x_{<t})\ln\dfrac{\mu(a\mid x_{<t})}{\xi(a\mid x_{<t})}$.
Show that $d_t\ge0$ even though $\xi(\cdot\mid x_{<t})$ may sum to less than 1. Then
show that $\sum_{t\le n}\mathbb E_\mu[d_t]\le\ln(1/w_\mu)$ for every $n$, and say
what follows as $t\to\infty$.

**(d)** Evaluate the bound for an environment with a 1000-bit description, in nats and
in bits. How does it compare with the cost of simply being told $\mu$?

<details><summary>Hints</summary>

(a) Every term in the sum is non-negative.

(b) Write each conditional as a ratio and let the product telescope. Note $\xi(\epsilon)\le1$
and $\mu(\epsilon)=1$. Then take $-\ln$ of (a).

(c) Use $\ln u\le u-1$ with $u=\xi(a\mid x_{<t})/\mu(a\mid x_{<t})$. For the sum, note
that the conditional expectation of the $t$-th regret term, given $x_{<t}$, is exactly
$d_t(x_{<t})$.
</details>

### 3.C. What the theorem is not

- **Not computable.** $M$ is only lower semicomputable. Every practical use is a
  computable surrogate that restricts the class or charges for runtime, for example
  MDL with a fixed code class, resource-bounded priors such as Schmidhuber's speed
  prior, context-tree weighting, and AIXI$tl$. Rathmanner and Hutter §9 surveys such
  approximations.
- **Not a probability measure.** It is a semimeasure, because mass leaks to programs
  that stop printing.
- **Not about functions.** $M$ is a prior over sequences. Supervised learning has
  inputs, so to apply it you must choose an encoding: interleave pairs $(x_i,y_i)$ into
  one sequence, or enumerate a finite input set $x_1,\dots,x_N$ and write a function as
  the string $f(x_1)\cdots f(x_N)$.
- **Not machine-free at finite $n$.** $K_U(\mu)$ depends on $U$ up to an additive
  constant (Exercise 2.B.i(b)). For a finite data budget that constant can dominate.
- **Not approximable by anything simple.** See Legg's reading and Exercise 3.C.i below.

#### 3.C.i. Diagonalize a predictor

**[Is there an elegant universal theory of prediction?](https://arxiv.org/abs/cs/0606070)**, Shane Legg, 2006 · *optional; core for this exercise*

For computable predictors of computable sequences: every predictor is defeated by a sequence of about its own complexity; predictors for all sequences up to complexity $n$ exist but must themselves have complexity about $n$; and beyond some $n$ no such predictor can be proved to work. Short, with each proof about a page; this exercise follows Lemmas 3.2–3.3 and Theorem 4.3.

A *predictor* is a computable map $p$ from finite binary strings to $\{0,1\}$. It
learns $\omega=\omega_1\omega_2\dots$ if there is an $m$ with
$p(\omega_{1:n})=\omega_{n+1}$ for all $n\ge m$ (Legg's Definition 2.4). For an
infinite computable sequence, $K(\omega)$ is the length of a shortest program that
outputs $\omega$ bit by bit on a monotone universal machine (as in §3.A); likewise
$K(p)$ is the length of a shortest program computing the predictor $p$.

**(a)** Given $p$, construct a computable sequence $\omega$ such that
$p(\omega_{1:n})\ne\omega_{n+1}$ for every $n$, and show $K(\omega)\le K(p)+c$ for a
constant $c$ independent of $p$.

**(b)** Deduce that any predictor that learns every computable sequence of complexity
at most $n$ has $K(p)\ge n-c$.

**(c)** Apply (a) to a trained language model with greedy decoding, viewed as a
deterministic next-token predictor. What sequence defeats it, and how complex is that
sequence? What is the analogue if the model is scored by log-loss?

**(d)** Legg's Lemma 3.2 says there is a single simple predictor that learns sequences
of arbitrarily high complexity. Find one, and reconcile it with (b).

<details><summary>Hints</summary>

(a) Let the sequence do the opposite of whatever $p$ predicts.

(b) Apply (a) to such a predictor. The defeating sequence cannot be in the class it learns.

(d) Think of sequences that are complicated only at the start.
</details>

### 3.D. A bridge to networks

**[Deep learning generalizes because the parameter-function map is biased towards simple functions](https://arxiv.org/abs/1805.08522)**, Guillermo Valle-Pérez, Chico Q. Camargo and Ard A. Louis, ICLR 2019 · *optional*

Argues that the prior over functions induced by random parameters is exponentially biased toward functions of low (proxy) complexity, and builds a PAC-Bayes bound on this prior that is well below chance on binarized MNIST and Fashion-MNIST and near chance on binarized CIFAR-10. The simplicity bound is one-sided and uses a computable proxy for $K$; §7 tests whether SGD samples like the posterior.

<a id="wb4-section-4"></a>

## 4. Program synthesis and the circuit prior

Two accounts that make the simplicity preference concrete: a hypothesis that SGD
searches for short programs, and a theorem that Bayesian fixed-depth MLPs prefer small
circuits.

### 4.A. Readings

**[Deep learning as program synthesis](https://www.lesswrong.com/posts/Dw8mskAvBX37MxvXo/deep-learning-as-program-synthesis-1)**, Zach Furman, January 2026 (LessWrong) · *core*

Argues, as a working hypothesis, that deep learning succeeds because SGD performs a tractable form of program synthesis: a search for simple, compositional algorithms that explain the data, analogous to but far cheaper than Solomonoff induction. Reverse-engineered circuits (grokking, curve detectors, induction heads) are the direct evidence, three "paradoxes" (approximation, generalization, convergence) the indirect evidence, and the representation problem and the search problem are named as what a formal version must solve.

**[A circuit prior in NN-bayes](https://www.lesswrong.com/posts/SqDHeuycNkERurtSc/a-circuit-prior-in-nn-bayes)**, Kaarel Hänni and Dmitry Vaintrob, August 2026 (LessWrong post linking [talk slides](https://kaarelh.github.io/ai/nnbayes-generalization.pdf), "A generalization result for overparametrized neural net Bayesian learning", Iliad 2026)

Shows, in talk slides with proof appendices (a full write-up is forthcoming), that Bayesian learning with an overparametrized fixed-depth MLP, under a strong-prior Gaussian scaling, has a circuit prior: a Boolean target of fixed-depth circuit complexity $C$ is learned from $\mathrm{poly}(d,C)$ samples at any width $N\ge C$, in the sense that the posterior's average probability on the correct label exceeds 99.9%. Slides 10–12 are a warm-up ("Solomonoff function induction") and 15–17 the setting and theorem; slide 34 contrasts this scaling with NNGP, which it describes as needing $\sim d^k$ samples for $k$-sparse parity, and Exercise 4.B.i derives that kernel count alongside a counting learner's $k\log_2d$.

### 4.B. What a prior buys on sparse parity

#### 4.B.i. Sparse parity: counting learner versus kernel learner

Inputs $x$ are uniform on $\{\pm1\}^d$. The target is
$\chi_S(x)=\prod_{i\in S}x_i$ for an unknown set $S$ with $|S|=k$, and labels are
noiseless. Write $N=\binom dk$ for the number of candidate targets.

**(a) A learner with a short-description prior.** Consider the learner that outputs
any $k$-sparse parity consistent with the $n$ training examples. Show that two
distinct parities disagree on exactly half of all inputs. Deduce that, with
probability at least $1-\delta$, the only consistent parity is the target once
$n\ge\log_2N+\log_2(1/\delta)$, and that $\log_2N\le k\log_2d$.

**(b) A fixed-kernel learner.** Kernel ridge regression, minimum-norm kernel
interpolation and the Gaussian-process posterior mean all output a function of the form
$\hat f=\sum_{i=1}^n\alpha_iK(x_i,\cdot)$, so $\hat f$ lies in
$V=\mathrm{span}\{K(x_i,\cdot)\}$. Note that $V$ depends on the inputs but not on the
labels. Working in $L^2$ of the uniform distribution, show that
$\sum_{|S|=k}\lVert P_V\chi_S\rVert^2\le n$, where $P_V$ is orthogonal projection onto
$V$. Deduce that, averaged over a uniformly random target $S$,
$\mathbb E_S\lVert\hat f-\chi_S\rVert^2\ge1-n/N$.

**(c)** Put numbers in for $d=100$, $k=5$. Then say what each result does and does not
tell you about networks trained by SGD.

<details><summary>Hints</summary>

(a) $\chi_S\chi_T=\chi_{S\triangle T}$, and a non-constant parity has mean zero. Then use
a union bound over the $N-1$ wrong parities.

(b) The parities $\chi_S$ are orthonormal in $L^2$ of the uniform distribution. Take an
orthonormal basis $e_1,\dots,e_r$ of $V$ ($r\le n$), expand
$\lVert P_V\chi_S\rVert^2=\sum_j\langle\chi_S,e_j\rangle^2$, swap the sums, and use
Bessel's inequality. Then $\lVert\hat f-\chi_S\rVert^2\ge\lVert\chi_S-P_V\chi_S\rVert^2=1-\lVert P_V\chi_S\rVert^2$.
</details>

<a id="wb4-section-5"></a>

## 5. Hierarchical representations

A different answer puts the structure in the target rather than in the learner's
prior.

### 5.A. Compositional targets

A depth-matched network approximates a binary tree of 2-variable functions with
$\sim(d-1)\varepsilon^{-2/r}$ units versus $\varepsilon^{-d/r}$ for a shallow one (an
existence result: Furman's "paradox of approximation").
[Source and smoothness assumptions: Poggio et al., Theorems 1–2.](https://arxiv.org/html/1611.00740)

### 5.B. A solvable learning model: the Random Hierarchy Model

**[How deep neural networks learn compositional data: the Random Hierarchy Model](https://arxiv.org/abs/2307.02129)**, Francesco Cagnetta, Leonardo Petrini, Umberto M. Tomasini, Alessandro Favero and Matthieu Wyart, 2024 (*Phys. Rev. X* 14, 031001) · *core*

On a synthetic family of hierarchical tasks, deep networks trained by gradient descent need a number of samples polynomial in the input dimension, about $n_c m^L$, while shallow networks need a number exponential in it. That sample size coincides with the onset of synonym-invariant representations and with the point where low-level feature–label correlations become detectable; removing those correlations makes the task exponentially hard even for deep networks. §I defines the model, §II gives the shallow-versus-deep scaling, and §III the correlation argument, ending with the uncorrelated control.

The model ingredients the exercise below uses:

- **Vocabulary and patches.** Each level has $v$ symbols. Each symbol at level $\ell$
  is rewritten as a patch of $s$ symbols at level $\ell-1$.
- **Synonyms.** Each symbol has $m$ equivalent rewrites ("synonyms"), chosen at random
  once and then fixed, with $m\le v^{s-1}$ so that rewrites are unambiguous.
- **Generation.** A class label (one of $n_c$) is rewritten $L$ times, which gives an
  input of length $d=s^L$.

Deep CNNs trained by gradient descent need about $P^*\simeq n_c\,m^L$ samples.

#### 5.B.i. Where does the learner's simplicity enter?

Estimate how many samples an idealized learner with a program-length prior would need
on the Random Hierarchy Model, and compare it with $P^*\simeq n_cm^L$ as $L$ grows.
What does the comparison suggest about what explains the observed $P^*$?

<details><summary>Hint</summary>

Count the bits needed to write down all the rules, level by level. In the warm-up of
the circuit-prior slides (§4.A, slides 10–12), a Bayesian learner over programs needs
a number of samples linear in the target's description length; also, each label
carries at most $\log_2n_c$ bits.
</details>

<a id="wb4-section-6"></a>

## 6. The accounts side by side

| prior over functions | where in this notebook | sample cost, $k$-sparse parity on $\{\pm1\}^d$ | correlated vs uncorrelated hierarchy (§5.B) |
|---|---|---|---|
| program length, $2^{-\ell(f)}$ | §2.B, §3; Furman (§4.A) | $\approx k\log_2d$, information-theoretic (Exercise 4.B.i(a)) | rules equally short, so rated equally hard |
| circuit size at fixed depth, $C_L(f)$ | Hänni–Vaintrob (§4.A) | $\mathrm{poly}(d,C)$, for Bayesian learning at any width $N\ge C$ | not addressed here |
| kernel norm, $e^{-\lVert f\rVert_{\mathcal H}^2/2}$ (NNGP, lazy/NTK; kernel-ridge learning curves in WB5 §7) | Exercise 4.B.i(b) | $\gtrsim\binom dk$ for any fixed kernel | not addressed here |
| *for comparison:* deep networks trained by gradient methods | Exercise 4.B.i(c), §5.B | time believed to grow like $d^{\Omega(k)}$ (statistical-query-style bounds) | polynomial ($\approx n_cm^L$) vs exponential in $d$ |

SLT describes posterior concentration in parameter space (§2.A's volume, done correctly
for singular models), not a fourth prior; see the SLT day.

## What carries forward

- **WB5 §2–3.** Different architectures learning similar representations is the "paradox of convergence" that program synthesis claims to explain; WB5 §2 is about how such similarity is measured and §3 about why it might arise. Linear and superposed feature representations are the subject of WB5 §4–5.
- **WB2 §7–8.** Evidence, Solomonoff and the circuit-prior theorem are statements about Bayesian predictors. Transferring these guarantees to a particular trained network requires an argument linking the theorem's predictor to that network. Posterior sampling is one possible link, and WB2 §8 tests whether SGD's stationary covariance on a quadratic basin matches a tempered posterior's; PAC-Bayes itself does not require the training algorithm to sample a Bayesian posterior.
- **WB3 §3.** Cross-entropy is code length, so a language model is a compressor. That is MDL in practice, and §2–3 here are its theory.
- **WB6.** One of its readings asks whether circuits are the right unit of explanation, which both program synthesis and the circuit prior assume.

## Further study

- Vaintrob and Hänni, ["Width-robust learnability in mean-field Bayesian neural networks"](https://arxiv.org/abs/2607.05735) (2026), for the mean-field (infinite-width) counterpart of the circuit-prior result.